In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
base_dir = Path(
    "/Volumes/shared/pyne_group/Shared/AFM_Data/Plasmids/pICoZ/20260105_20251031_20251107_combined_picoz_dataset/202608XX-response-to-reviewers"
)
assert base_dir.exists()

child_directories = [directory for directory in base_dir.iterdir() if directory.is_dir()]
print(f"Found {len(child_directories)} child directories in {base_dir.name}:")
for directory in child_directories:
    print(f" - {directory.name}")
# remove "archive" from the list of child directories
child_directories = [directory for directory in child_directories if directory.name != "archive"]

datasets = {}
for child_directory in child_directories:
    # see if there is a file called "grain_statistics.csv"
    grainstats_file = child_directory / "grain_statistics.csv"
    if grainstats_file.exists():
        print(f"Found grain_statistics.csv in {child_directory.name}")
        grain_stats = pd.read_csv(grainstats_file)
        print(f"Loaded {len(grain_stats)} rows of grain statistics from {grainstats_file}")
        # calculate residuals from the expected contour length
        grain_stats["contour_length_residuals"] = np.abs(grain_stats["total_contour_length"] - 434e-9)
        grain_stats["contour_length_residuals"] *= 1e9  # convert to nm
        datasets[child_directory.name] = {
            "grain_stats": grain_stats,
            "xtick_label": child_directory.name,
        }
    else:
        print(f"No grain_statistics.csv found in {child_directory.name}")

order = [
    "output_classical_optimal",
    "output_catsnet_p5_c05",
    "output_classical_low_height",
    "output_classical_high_height",
    "output_classical_no_end_joining",
    "output_classical_no_pruning",
    "output_classical_no_height_bias",
    "output_classical_no_trace_smoothing",
    "output_classical_everything_off",
]

if any(dataset_name not in datasets for dataset_name in order):
    missing_datasets = [dataset_name for dataset_name in order if dataset_name not in datasets]
    raise ValueError(f"Missing datasets: {missing_datasets}")

# reorder the datasets dictionary according to the order list
datasets = {dataset_name: datasets[dataset_name] for dataset_name in order}

datasets[list(datasets.keys())[0]]["grain_stats"].head()

# plot stats between them

In [ ]:
colours = [
    "#000",
    "#E79F00",
    "#57B4E9",
    "#019E73",
    "#F0E441",
    "#0072B2",
    "#D55E00",
    "#CC79A7",
    "#AAA",
]
columns_to_plot = {
    "total_contour_length": {
        "scaling_factor": 1e9,
        "expected_value": 434,
        "title": "Total Contour Length (nm)",
    },
    "grain_endpoints": {
        "scaling_factor": 1,
        "expected_value": None,
        "title": "Grain Endpoints",
    },
    "contour_length_residuals": {
        "scaling_factor": 1,
        "expected_value": None,
        "title": "Contour Length Residuals (nm)",
    },
}
for column_to_plot, column_properties in columns_to_plot.items():
    column_scaling_factor = column_properties["scaling_factor"]
    expected_value = column_properties["expected_value"]
    title = column_properties["title"]
    for dataset_name, dataset in datasets.items():
        grain_stats = dataset["grain_stats"]
        if column_to_plot in grain_stats.columns:
            print("\n\n")
            print(f"Dataset: {dataset_name}, {title} stats:")
            print(grain_stats[column_to_plot].describe())
        else:
            print(f"Dataset: {dataset_name} does not have column {column_to_plot}")
            print(f"Columns available: {grain_stats.columns.tolist()}")

    # plot a stripplot of the stat for each dataset
    fig, ax = plt.subplots(figsize=(6, 4))
    colour_index = 0
    for dataset_name, dataset in datasets.items():
        grain_stats = dataset["grain_stats"]
        grain_stats_stats = {}
        grain_stats_stats["mean"] = grain_stats[column_to_plot].mean()
        grain_stats_stats["std"] = grain_stats[column_to_plot].std()
        grain_stats_stats["min"] = grain_stats[column_to_plot].min()
        grain_stats_stats["max"] = grain_stats[column_to_plot].max()
        grain_stats_stats["median"] = grain_stats[column_to_plot].median()
        print()
        print(f"Dataset: {dataset_name}, {title} stats:")
        print(grain_stats_stats)
        number_of_grains = len(grain_stats)
        # add number of grains to the x tick label
        tick_label = dataset_name.replace("output_classical_", "")
        if "catsnet" in tick_label:
            tick_label = "catsnet"
        tick_label = tick_label.replace("_", " ")
        tick_label = tick_label + f"\n(n={number_of_grains})"
        dataset["xtick_label"] = tick_label
        if column_to_plot in grain_stats.columns:
            sns.stripplot(
                x=[dataset["xtick_label"]] * len(grain_stats),
                y=grain_stats[column_to_plot] * column_scaling_factor,
                ax=ax,
                jitter=True,
                alpha=0.5,
                color=colours[colour_index % len(colours)],
            )
            # draw a boxplot on top of the stripplot
            sns.boxplot(
                x=[dataset["xtick_label"]] * len(grain_stats),
                y=grain_stats[column_to_plot] * column_scaling_factor,
                ax=ax,
                showcaps=True,
                boxprops={"facecolor": "None"},
                showfliers=False,
                whiskerprops={"linewidth": 2},
            )
        colour_index += 1
    # add a horizontal line for the expected value
    if expected_value is not None:
        ax.axhline(expected_value, color="red", linestyle="--", label=f"Expected value: {expected_value}")
    ax.set_ylabel(title)
    # rotate x-axis labels for better readability
    plt.xticks(rotation=90)
    ax.legend()
    # save the figure
    plt.tight_layout()
    plt.savefig(base_dir / f"{column_to_plot}_comparison.png", dpi=300)
    plt.show()
    # save the stats for this column to a csv file
    stats_df = pd.DataFrame(
        {
            "dataset_name": [],
            "mean": [],
            "std": [],
            "min": [],
            "max": [],
            "median": [],
        }
    )
    for dataset_name, dataset in datasets.items():
        grain_stats = dataset["grain_stats"]
        if column_to_plot in grain_stats.columns:
            stats_df = pd.concat(
                [
                    stats_df,
                    pd.DataFrame(
                        {
                            "dataset_name": [dataset_name],
                            "mean": [grain_stats[column_to_plot].mean()],
                            "std": [grain_stats[column_to_plot].std()],
                            "min": [grain_stats[column_to_plot].min()],
                            "max": [grain_stats[column_to_plot].max()],
                            "median": [grain_stats[column_to_plot].median()],
                        }
                    ),
                ],
                ignore_index=True,
            )
    stats_df.to_csv(base_dir / f"{column_to_plot}_stats.csv", index=False)
    print(f"Saved stats for {column_to_plot} to {base_dir / f'{column_to_plot}_stats.csv'}")